# 04b - StrongREJECT / WildGuard preflight (blocker B1)

Load both judges at 8-bit on toy pairs, print VRAM + versions. Branch per analysis_plan.md §10 row 10 on failure. **No full run here** - short session, not the 240-270 min window.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd'   # exact commit this session runs against

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

## 3. Install dependencies, check GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized/8-bit
# loads (see 04b). Matches colab_unified_{analysis,training}.ipynb.
!pip uninstall -y torchao || true
!nvidia-smi

## 4. Persistent storage (results/ + HF cache bound to Drive)

In [ ]:
# Bind results/ + the HF weight cache to a persistent Drive folder so this
# session's work survives a Colab disconnect and a fresh VM resumes it. One
# line - the logic + tests live in src/colab_persist.py. Idempotent, and safe
# even if you have already done work on the ephemeral results/ (it merges that
# into Drive first). Override the Drive root with the DPO_DRIVE_ROOT env var;
# pass persist_hf_cache=False to keep the ~5 GB HF cache off Drive.
from src.colab_persist import bind, status_line
info = bind()                     # or: bind(persist_hf_cache=False)
print(status_line(info))
!python -m src.analysis.v2_pipeline status

## 5. Load StrongREJECT fine-tuned Gemma-2B

In [ ]:
from src.analysis.behavioral_judges import LazyModelJudge, parse_strongreject_output
sr = LazyModelJudge('strong_reject', 'dsbowen/strong_reject'); print(sr.try_load())

## 6. Load WildGuard

In [ ]:
wg = LazyModelJudge('wildguard', 'allenai/wildguard'); print(wg.try_load())

## 7. Pin `model_id@revision` into docs/audit/analysis_plan.md (B3)

In [ ]:
# record the resolved revision hashes here

## 8. Parser smoke test (no model needed)

In [ ]:
print(parse_strongreject_output('1.b 0\n2.b 4\n3.b 3\n'))